# AML Analytical Dashboard — Data Computation

This notebook computes:
1. Edge risk scores (source/target risk mapping)
2. Money flow per node (incoming/outgoing/net)
3. Loss vs savings financial analysis
4. Node risk profiles

**No PNGs generated** — visualization handled by the chart exporter service.
**Pre-computed scores** loaded from notebook 10 output (no redundant model.predict).

In [ ]:
# Setup and imports
import os
import json
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define paths
BASE_PATH = os.path.dirname(os.path.abspath("__file__"))
TRAINING_DATA_PATH = os.path.join(BASE_PATH, "training_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "output")
MODELS_PATH = os.path.join(BASE_PATH, "models")

# Override paths when running via pipeline (artifacts_dir injected by papermill)
try:
    if artifacts_dir:
        TRAINING_DATA_PATH = os.path.join(artifacts_dir, "data")
        OUTPUT_PATH = os.path.join(artifacts_dir, "data")
        MODELS_PATH = os.path.join(artifacts_dir, "models")
except NameError:
    pass

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Demo configuration - Assumption rates for loss/savings calculations
DEMO_CONFIG = {
    'avg_loss_per_undetected_aml': 0.15,
    'investigation_cost_per_alert': 500,
    'false_positive_cost': 200,
    'regulatory_fine_multiplier': 3.0,
    'recovery_rate_detected': 0.70,
    'currency': 'USD'
}

print("Dashboard data pipeline initialized!")

In [ ]:
# Load all data (using pre-computed scores from notebook 10)
print("Loading data...")

# Load edges (transactions)
edges_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "edges_td.csv"))
print(f"  Transactions: {len(edges_df):,}")

# Load nodes (parties)
nodes_df = pd.read_csv(os.path.join(TRAINING_DATA_PATH, "node_td.csv"))
print(f"  Nodes: {len(nodes_df):,}")

# Load node embeddings (already enriched with anomaly_score, is_anomaly, risk_score by notebook 10)
node_embeddings = pd.read_parquet(os.path.join(OUTPUT_PATH, "node_embeddings_fg.parquet"))
print(f"  Node embeddings: {len(node_embeddings):,}")

# Verify pre-computed scores exist
assert 'anomaly_score' in node_embeddings.columns, "anomaly_score missing — run notebook 10 first"
assert 'is_anomaly' in node_embeddings.columns, "is_anomaly missing — run notebook 10 first"
assert 'risk_score' in node_embeddings.columns, "risk_score missing — run notebook 10 first"

# Load threshold
model_dirs = [d for d in os.listdir(MODELS_PATH) if d.startswith('gan_anomaly_')]
latest_model_dir = os.path.join(MODELS_PATH, sorted(model_dirs)[-1])
threshold = np.load(os.path.join(latest_model_dir, "threshold.npy"))

print(f"  Using pre-computed scores from notebook 10 (no model.predict needed)")
print(f"  Anomaly threshold: {threshold:.6f}")
print(f"  Anomalies: {node_embeddings['is_anomaly'].sum():,} / {len(node_embeddings):,}")
print(f"\nData loading complete!")

In [ ]:
# Prepare transaction-level risk data
# Map node risk to edges
node_risk_dict = node_embeddings.set_index('id')['risk_score'].to_dict()
node_anomaly_dict = node_embeddings.set_index('id')['is_anomaly'].to_dict()

# Calculate edge risk (max of source and target risk)
edges_df['source_risk'] = edges_df['source'].map(node_risk_dict).fillna(0)
edges_df['target_risk'] = edges_df['target'].map(node_risk_dict).fillna(0)
edges_df['edge_risk'] = edges_df[['source_risk', 'target_risk']].max(axis=1)
edges_df['source_anomaly'] = edges_df['source'].map(node_anomaly_dict).fillna(False)
edges_df['target_anomaly'] = edges_df['target'].map(node_anomaly_dict).fillna(False)
edges_df['is_suspicious'] = edges_df['source_anomaly'] | edges_df['target_anomaly']

# Map node types onto edges
node_type_dict = nodes_df.set_index('id')['type'].to_dict()
edges_df['source_type'] = edges_df['source'].map(node_type_dict).fillna(-1).astype(int)
edges_df['target_type'] = edges_df['target'].map(node_type_dict).fillna(-1).astype(int)

# Calculate money flow per node
outgoing = edges_df.groupby('source').agg({'base_amt': 'sum', 'tran_id': 'count'}).rename(
    columns={'base_amt': 'outgoing_amt', 'tran_id': 'outgoing_count'})
incoming = edges_df.groupby('target').agg({'base_amt': 'sum', 'tran_id': 'count'}).rename(
    columns={'base_amt': 'incoming_amt', 'tran_id': 'incoming_count'})

node_money = node_embeddings[['id', 'anomaly_score', 'is_anomaly', 'risk_score']].copy()
if 'is_sar' in node_embeddings.columns:
    node_money['is_sar'] = node_embeddings['is_sar']
node_money = node_money.merge(outgoing, left_on='id', right_index=True, how='left')
node_money = node_money.merge(incoming, left_on='id', right_index=True, how='left')
node_money = node_money.fillna(0)
node_money['total_volume'] = node_money['outgoing_amt'] + node_money['incoming_amt']
node_money['total_transactions'] = node_money['outgoing_count'] + node_money['incoming_count']
node_money['net_flow'] = node_money['incoming_amt'] - node_money['outgoing_amt']

# Add node type
node_money = node_money.merge(nodes_df[['id', 'type']], on='id', how='left')

# Save enriched edges and node money flow to parquet
edges_out = os.path.join(OUTPUT_PATH, 'edges_enriched.parquet')
edges_df.to_parquet(edges_out, index=False)
print(f"Saved enriched edges to {edges_out} ({len(edges_df)} rows)")

money_out = os.path.join(OUTPUT_PATH, 'node_money_flow.parquet')
node_money.to_parquet(money_out, index=False)
print(f"Saved node money flow to {money_out} ({len(node_money)} rows)")

print("Edge risk and money flow data prepared and saved!")

---
# Executive Summary KPIs
---

In [ ]:
# Executive Summary KPIs (visualization handled by chart exporter)
suspicious_vol = edges_df[edges_df['is_suspicious']]['base_amt'].sum()
normal_vol = edges_df[~edges_df['is_suspicious']]['base_amt'].sum()

print("Executive Summary KPIs:")
print(f"  Total Transactions:   {len(edges_df):,}")
print(f"  Total Volume:         ${edges_df['base_amt'].sum():,.0f}")
print(f"  Anomalies Detected:   {node_embeddings['is_anomaly'].sum():,}")
print(f"  Detection Rate:       {100*node_embeddings['is_anomaly'].mean():.1f}%")
print(f"  Suspicious Volume:    ${suspicious_vol:,.0f}")
print(f"  Normal Volume:        ${normal_vol:,.0f}")

---
# Loss vs Savings Analysis
---

In [5]:
# Loss vs Savings Calculator
def calculate_loss_savings():
    """Calculate potential losses avoided and operational costs"""
    
    # Suspicious transaction value
    suspicious_txn_value = edges_df[edges_df['is_suspicious']]['base_amt'].sum()
    normal_txn_value = edges_df[~edges_df['is_suspicious']]['base_amt'].sum()
    
    # Number of alerts
    n_anomalies = node_embeddings['is_anomaly'].sum()
    n_normal = len(node_embeddings) - n_anomalies
    
    # Estimate true positives and false positives (using SAR labels if available)
    if 'is_sar' in node_embeddings.columns:
        true_positives = ((node_embeddings['is_sar'] == 1) & (node_embeddings['is_anomaly'])).sum()
        false_positives = ((node_embeddings['is_sar'] == 0) & (node_embeddings['is_anomaly'])).sum()
        false_negatives = ((node_embeddings['is_sar'] == 1) & (~node_embeddings['is_anomaly'])).sum()
        true_negatives = ((node_embeddings['is_sar'] == 0) & (~node_embeddings['is_anomaly'])).sum()
    else:
        # Estimate based on industry averages (assume 10% of anomalies are true AML)
        est_true_positive_rate = 0.10
        true_positives = int(n_anomalies * est_true_positive_rate)
        false_positives = n_anomalies - true_positives
        false_negatives = int(n_normal * 0.01)  # Assume 1% of normals are missed AML
        true_negatives = n_normal - false_negatives
    
    # Financial calculations
    cfg = DEMO_CONFIG
    
    # Potential loss from detected AML (avoided)
    avg_suspicious_txn = suspicious_txn_value / max(n_anomalies, 1)
    potential_loss_detected = true_positives * avg_suspicious_txn * cfg['avg_loss_per_undetected_aml']
    recovered_amount = potential_loss_detected * cfg['recovery_rate_detected']
    
    # Potential loss from undetected AML (not avoided)
    avg_normal_txn = normal_txn_value / max(n_normal, 1)
    potential_loss_undetected = false_negatives * avg_normal_txn * cfg['avg_loss_per_undetected_aml']
    regulatory_fine_risk = potential_loss_undetected * cfg['regulatory_fine_multiplier']
    
    # Operational costs
    investigation_cost = n_anomalies * cfg['investigation_cost_per_alert']
    false_positive_cost = false_positives * cfg['false_positive_cost']
    total_operational_cost = investigation_cost + false_positive_cost
    
    # Net savings
    gross_savings = recovered_amount
    net_savings = gross_savings - total_operational_cost
    
    return {
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'true_negatives': true_negatives,
        'suspicious_txn_value': suspicious_txn_value,
        'potential_loss_detected': potential_loss_detected,
        'recovered_amount': recovered_amount,
        'potential_loss_undetected': potential_loss_undetected,
        'regulatory_fine_risk': regulatory_fine_risk,
        'investigation_cost': investigation_cost,
        'false_positive_cost': false_positive_cost,
        'total_operational_cost': total_operational_cost,
        'gross_savings': gross_savings,
        'net_savings': net_savings
    }

loss_savings = calculate_loss_savings()
print("Loss/Savings analysis calculated!")

Loss/Savings analysis calculated!


In [ ]:
# Loss vs Savings results (visualization handled by chart exporter)
ls = loss_savings

# Detection performance
precision = ls['true_positives'] / max(ls['true_positives'] + ls['false_positives'], 1)
recall = ls['true_positives'] / max(ls['true_positives'] + ls['false_negatives'], 1)
f1 = 2 * (precision * recall) / max(precision + recall, 0.0001)

print("Loss vs Savings Analysis:")
print(f"  True Positives:       {ls['true_positives']:,}")
print(f"  False Positives:      {ls['false_positives']:,}")
print(f"  False Negatives:      {ls['false_negatives']:,}")
print(f"  True Negatives:       {ls['true_negatives']:,}")
print(f"  Precision:            {precision:.2%}")
print(f"  Recall:               {recall:.2%}")
print(f"  F1 Score:             {f1:.2%}")
print()
print(f"  Suspicious Txn Value: ${ls['suspicious_txn_value']:,.0f}")
print(f"  Loss Avoided:         ${ls['potential_loss_detected']:,.0f}")
print(f"  Amount Recovered:     ${ls['recovered_amount']:,.0f}")
print(f"  Investigation Cost:   ${ls['investigation_cost']:,.0f}")
print(f"  FP Cost:              ${ls['false_positive_cost']:,.0f}")
print(f"  Net Savings:          ${ls['net_savings']:,.0f}")

# Save financial metrics as JSON
import json
metrics_out = os.path.join(OUTPUT_PATH, 'financial_metrics.json')
with open(metrics_out, 'w') as f:
    json.dump(ls, f, indent=2, default=str)
print(f"\nSaved financial metrics to {metrics_out}")

---
# Risk Network Summary
---

In [ ]:
# Risk network summary (visualization handled by chart exporter)
high_risk_edges = edges_df[edges_df['edge_risk'] > 0.75]
suspicious_edges = edges_df[edges_df['is_suspicious']]

print("Risk Network Summary:")
print(f"  Total edges:          {len(edges_df):,}")
print(f"  Suspicious edges:     {len(suspicious_edges):,} ({100*len(suspicious_edges)/len(edges_df):.1f}%)")
print(f"  High-risk edges:      {len(high_risk_edges):,} ({100*len(high_risk_edges)/len(edges_df):.1f}%)")
print(f"  Avg edge risk:        {edges_df['edge_risk'].mean():.4f}")
print(f"  Max edge risk:        {edges_df['edge_risk'].max():.4f}")

---
# Transaction Deep Dive Summary
---

In [ ]:
# Transaction deep dive summary (visualization handled by chart exporter)
risk_categories = pd.cut(edges_df['edge_risk'], bins=[0, 0.25, 0.5, 0.75, 1.0],
                         labels=['Low', 'Medium', 'High', 'Critical'])
risk_vol = edges_df.groupby(risk_categories)['base_amt'].agg(['sum', 'mean', 'count'])

print("Transaction Volume by Risk Category:")
print(risk_vol.to_string())

print(f"\nTop 10 highest-value transactions:")
top_txns = edges_df.nlargest(10, 'base_amt')[['source', 'target', 'base_amt', 'edge_risk', 'is_suspicious']]
print(top_txns.to_string(index=False))

---
# Node Risk Profiles Summary
---

In [ ]:
# Node risk profiles summary (visualization handled by chart exporter)
high_risk = node_money[node_money['risk_score'] > 0.75]
med_risk = node_money[(node_money['risk_score'] > 0.25) & (node_money['risk_score'] <= 0.75)]
low_risk = node_money[node_money['risk_score'] <= 0.25]

print("Node Risk Profiles:")
print(f"  Low risk (0-0.25):      {len(low_risk):,} nodes, avg vol ${low_risk['total_volume'].mean():,.0f}")
print(f"  Medium risk (0.25-0.75): {len(med_risk):,} nodes, avg vol ${med_risk['total_volume'].mean():,.0f}")
print(f"  High risk (0.75-1.0):   {len(high_risk):,} nodes, avg vol ${high_risk['total_volume'].mean():,.0f}")

print(f"\nTop 10 highest risk nodes:")
top10 = node_money.nlargest(10, 'risk_score')[['id', 'risk_score', 'total_volume', 'total_transactions']]
print(top10.to_string(index=False))

---
# Pipeline Summary
---

In [ ]:
# Data pipeline complete — no interactive widgets needed
print("All analytical data computed and saved.")

---
# Dashboard Summary Report
---

In [ ]:
# Final Dashboard Summary
print("\n" + "="*80)
print(" AML ANALYTICAL DASHBOARD - SUMMARY REPORT ")
print("="*80)

print(f"\n{'DATASET OVERVIEW':^40}")
print("-"*40)
print(f"  Total Nodes:              {len(node_embeddings):>12,}")
print(f"  Total Transactions:       {len(edges_df):>12,}")
print(f"  Total Value:              ${edges_df['base_amt'].sum():>11,.0f}")

print(f"\n{'DETECTION RESULTS':^40}")
print("-"*40)
print(f"  Anomalies Detected:       {node_embeddings['is_anomaly'].sum():>12,}")
print(f"  Detection Rate:           {100*node_embeddings['is_anomaly'].mean():>11.1f}%")
print(f"  Suspicious Transactions:  {edges_df['is_suspicious'].sum():>12,}")

print(f"\n{'FINANCIAL IMPACT':^40}")
print("-"*40)
print(f"  Suspicious Volume:        ${ls['suspicious_txn_value']:>11,.0f}")
print(f"  Potential Loss Avoided:   ${ls['potential_loss_detected']:>11,.0f}")
print(f"  Amount Recovered:         ${ls['recovered_amount']:>11,.0f}")
print(f"  Operational Cost:         ${ls['total_operational_cost']:>11,.0f}")
print(f"  Net Savings:              ${ls['net_savings']:>11,.0f}")

print(f"\n{'DATA OUTPUTS SAVED':^40}")
print("-"*40)
print(f"  - edges_enriched.parquet (edge risk, is_suspicious, source/target types)")
print(f"  - node_money_flow.parquet (money flow per node)")
print(f"  - financial_metrics.json (loss/savings analysis)")

print("\n" + "="*80)
print(" DASHBOARD DATA PIPELINE COMPLETE ")
print("="*80)